# Mô hình dự đoán kết quả trận đấu giải Premier League (Logistic Regression)

## 1. Giới thiệu tổng quan
Dự đoán kết quả trận đấu bóng đá Ngoại hạng Anh (Premier League) là một bài toán phân loại đa lớp phức tạp. Dự đoán kết quả dựa trên các chỉ số phong độ tiền trận đấu (pre-match features) giúp đảm bảo không bị rò rỉ dữ liệu (data leakage).

## 2. Phương pháp luận (Multinomial Logistic Regression)
Mô hình sử dụng thuật toán **Multinomial Logistic Regression** (Hồi quy Logistic đa thức) để phân loại kết quả trận đấu thành 3 nhãn riêng biệt:
- **`H` (Home Win):** Đội nhà thắng
- **`A` (Away Win):** Đội khách thắng
- **`D` (Draw):** Hai đội hòa nhau

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

model_df = pd.read_csv('ml_features_dataset.csv')
model_df = model_df.dropna().copy()
print(f"Kích thước tập dữ liệu đặc trưng: {model_df.shape}")

Kích thước tập dữ liệu đặc trưng: (2970, 7)


## 3. Ý nghĩa và Cấu trúc các Đặc trưng (Feature Architecture)

Tập dữ liệu đầu vào bao gồm 3 nhóm chỉ số quan trọng phản ánh phong độ gần đây và lịch sử đối đầu giữa hai đội:

### 3.1. Phong độ 5 trận gần nhất của Đội nhà (Recent Performance - Home Team)
- **`home_scored_last5`:** Số bàn thắng trung bình ghi được trong 5 trận gần nhất của đội nhà. Chỉ số này phản ánh hiệu suất và sức mạnh tấn công gần nhất của đội chủ nhà trước thời điểm trận đấu diễn ra.
- **`home_conceded_last5`:** Số bàn thua trung bình trong 5 trận gần nhất của đội nhà. Chỉ số này đánh giá độ chắc chắn và hiệu quả của hàng phòng ngự đội chủ nhà.

### 3.2. Phong độ 5 trận gần nhất của Đội khách (Recent Performance - Away Team)
- **`away_scored_last5`:** Số bàn thắng trung bình ghi được trong 5 trận gần nhất của đội khách. Chỉ số này thể hiện khả năng săn bàn và phong độ thi đấu trên sân khách của đội làm khách.
- **`away_conceded_last5`:** Số bàn thua trung bình trong 5 trận gần nhất của đội khách. Chỉ số này phản ánh mức độ dễ bị tổn thương của hàng thủ đội khách.

### 3.3. Lịch sử đối đầu trực tiếp (Past Head-to-Head Matchups - H2H)
- **`h2h_home_goals`:** Số bàn thắng trung bình mà đội nhà ghi được khi đối đầu trực tiếp với đội khách trong 5 lần gặp nhau gần nhất. Chỉ số này bắt bài yếu tố kỵ rơ hoặc lợi thế tâm lý đối đầu của đội chủ nhà.
- **`h2h_away_goals`:** Số bàn thắng trung bình mà đội khách ghi được trong 5 lần đối đầu gần nhất với đội nhà.

In [5]:
feature_cols = [
    'home_scored_last5', 'home_conceded_last5', 
    'away_scored_last5', 'away_conceded_last5', 
    'h2h_home_goals', 'h2h_away_goals'
]

X = model_df[feature_cols]
y = model_df['result']

## 4. Huấn luyện và Đánh giá Mô hình

- **Phân chia theo thời gian (Chronological Split):** Chia tập dữ liệu 80% huấn luyện và 20% kiểm thử theo đúng thứ tự thời gian để mô phỏng chính xác bài toán dự đoán các trận đấu tương lai.
- **Chuẩn hóa đặc trưng (`StandardScaler`):** Đưa các đặc trưng về phân phối chuẩn (mean=0, std=1) giúp thuật toán Hồi quy Logistic hội tụ nhanh hơn và các trọng số đóng góp công bằng.
- **Cân bằng lớp (`class_weight='balanced'`):** Xử lý hiện tượng mất cân bằng lớp (kết quả Hòa chiếm tỷ lệ thấp hơn Thắng/Thua), giúp mô hình nhận diện tốt hơn cửa Hòa.

In [6]:
split_idx = int(len(model_df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
clf.fit(X_train_scaled, y_train)

y_pred = clf.predict(X_test_scaled)
print(f"Độ chính xác tổng thể của mô hình (Accuracy): {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("Báo cáo phân loại chi tiết:")
print(classification_report(y_test, y_pred))

Độ chính xác tổng thể của mô hình (Accuracy): 50.34%
Báo cáo phân loại chi tiết:
              precision    recall  f1-score   support

           A       0.50      0.57      0.53       194
           D       0.31      0.21      0.25       139
           H       0.57      0.61      0.59       261

    accuracy                           0.50       594
   macro avg       0.46      0.46      0.46       594
weighted avg       0.49      0.50      0.49       594

